# IZQAR — IBM AML graph investigation

Add Input: `ealtman2019/ibm-transactions-for-anti-money-laundering-aml`. This validation aid inspects a bounded chronological slice of synthetic transfers without reading `Is Laundering`. It produces investigation leads, not confirmed laundering labels. The detailed JSON stays a private Kaggle output; IZQAR's API exposes only aggregate counts.

In [ ]:
from pathlib import Path
import json, subprocess, sys
EXPECTED_REVISION = '13913745a66fcf70c658cc7ab75ee301159db446'
root = Path('/kaggle/working/izqar_ibm_graph')
if root.exists():
    raise FileExistsError('Use a fresh session; refusing a stale code checkout')
subprocess.run(['git', 'clone', '--branch', 'feat/ml-pipelines-and-startup', 'https://github.com/Night011/izqar.git', str(root)], check=True)
subprocess.run(['git', 'checkout', '--detach', EXPECTED_REVISION], cwd=root, check=True)
actual_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=root, text=True).strip()
if actual_revision != EXPECTED_REVISION:
    raise RuntimeError('Required reviewed revision was not checked out')
matches = list(Path('/kaggle/input').rglob('HI-Small_Trans.csv'))
if len(matches) != 1:
    raise FileNotFoundError(f'Expected exactly one HI-Small_Trans.csv, got {len(matches)}')
data = matches[0]
print('Input:', data, 'bytes:', data.stat().st_size, 'code:', actual_revision)


In [ ]:
backend = root / 'backend'
report = Path('/kaggle/working/ibm_graph_report.json')
subprocess.run([sys.executable, str(backend / 'scripts' / 'analyze_ibm_graph.py'), '--data', str(data), '--output', str(report), '--max-rows', '500000', '--rapid-seconds', '900', '--min-degree', '3', '--max-results', '50'], cwd=backend, check=True)
detail = json.loads(report.read_text(encoding='utf-8'))
assert detail['rows_analyzed'] == 500000
assert detail['source'] == 'IBM AML HI-Small synthetic transfers'
run = Path('/kaggle/working/ibm_graph_run.json')
run.write_text(json.dumps({'expected_revision': EXPECTED_REVISION, 'actual_revision': actual_revision, 'dataset_input_file': data.name, 'dataset_input_bytes': data.stat().st_size, 'max_rows': 500000, 'label_used': False, 'report_contains_synthetic_account_leads': True}, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({'rows_analyzed': detail['rows_analyzed'], 'edge_count': detail['edge_count'], 'signals': {key: len(detail[key]) for key in ('fan_in', 'fan_out', 'rapid_transfer_chains', 'directed_three_node_cycles')}}, ensure_ascii=False, indent=2))
